# GACS Research Pipeline - Full Notebook

이 노트북은 GACS 연구 파이프라인 전체를 하나로 통합한 버전입니다.

## 실행 순서
1. **Part 1**: Dataset Builder - 비디오에서 장면 추출 및 감정 라벨링
2. **Part 2**: Video Generator - 무드 기반 비디오 생성
3. **Part 3**: YouTube Experiment - 유튜브 업로드 및 메트릭 수집

---

# 0. 환경 설정 (Setup)

In [ ]:
# Colab에서 실행시 주석 해제
# !pip install opencv-python scenedetect anthropic numpy pandas Pillow matplotlib tqdm sentence-transformers moviepy scikit-learn google-api-python-client google-auth-oauthlib scipy seaborn fpdf

In [ ]:
import base64
import hashlib
import json
import random
import shutil
import time
from dataclasses import asdict, dataclass, field
from datetime import datetime, timedelta
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import anthropic
import cv2
import numpy as np
import pandas as pd
from moviepy.editor import VideoFileClip, concatenate_videoclips
from moviepy.video.fx.all import fadein, fadeout
from scenedetect import ContentDetector, SceneManager, open_video
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from tqdm.notebook import tqdm

print("All dependencies loaded!")

In [ ]:
# API Key 설정
# os.environ['ANTHROPIC_API_KEY'] = 'your-api-key-here'

# 경로 설정
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RAW_VIDEOS_DIR = DATA_DIR / "raw_videos"
SCENES_DIR = DATA_DIR / "scenes"
ANNOTATIONS_DIR = DATA_DIR / "annotations"
EMBEDDINGS_DIR = DATA_DIR / "embeddings"
GENERATED_DIR = DATA_DIR / "generated"
GACS_OUTPUT_DIR = GENERATED_DIR / "gacs"
BASELINE_OUTPUT_DIR = GENERATED_DIR / "baseline"
EXPERIMENTS_DIR = DATA_DIR / "experiments"

# 디렉토리 생성
for d in [SCENES_DIR, ANNOTATIONS_DIR, EMBEDDINGS_DIR, GACS_OUTPUT_DIR, BASELINE_OUTPUT_DIR, EXPERIMENTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 설정값
CLAUDE_MODEL = "claude-sonnet-4-20250514"
SBERT_MODEL = "all-MiniLM-L6-v2"
API_RATE_LIMIT_DELAY = 1.0
PROMPT_VERSION = "v1.0"
SCENE_THRESHOLD = 27.0
MIN_SCENE_LENGTH = 15
DEFAULT_K_CLUSTERS = 5
TARGET_DURATION = 30
MIN_SCENES = 5
MAX_SCENES = 8
OUTPUT_FPS = 30
TRANSITION_DURATION = 0.5

print(f"Base directory: {BASE_DIR}")

In [ ]:
# Claude client 초기화
client = anthropic.Anthropic()

# SBERT 모델 로드
print("Loading SBERT model...")
sbert_model = SentenceTransformer(SBERT_MODEL)
print("SBERT model loaded!")

---
# Part 1: Dataset Builder
비디오에서 장면 추출, 키프레임 추출, Claude API로 감정 라벨링

---

In [ ]:
# Data Classes
@dataclass
class Scene:
    scene_id: str
    video_id: str
    scene_index: int
    start_frame: int
    end_frame: int
    start_time: float
    end_time: float
    duration: float
    keyframe_path: Optional[str] = None

@dataclass
class Annotation:
    image_id: str
    video_id: str
    scene_id: str
    mood_words: List[str]
    style_words: List[str]
    object_words: List[str]
    labeling_model: str
    prompt_version: str
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())

In [ ]:
# Claude API Prompts
MOOD_PROMPT = """You are an affective labeling engine.
List exactly five adjectives that best describe the emotional mood.
Rules:
- Only adjectives.
- No sentences.
- No explanations.
- No duplicates.
- Avoid generic words.
Output format:
adj1, adj2, adj3, adj4, adj5"""

STYLE_PROMPT = """You are a visual style classifier.
List exactly three short visual style descriptors.
Rules:
- Under 12 characters each.
- No emotions.
- No sentences.
Output format:
style1, style2, style3"""

OBJECT_PROMPT = """You are an object extraction system.
List 3 to 5 concrete object nouns visible.
Rules:
- Only nouns.
- No abstract concepts.
- No sentences.
Output format:
obj1, obj2, obj3, obj4, obj5"""

In [ ]:
# Part 1 Functions
def detect_scenes(video_path: Path, video_id: str) -> List[Scene]:
    if not video_path.exists():
        print(f"Video not found: {video_path}")
        return []

    video = open_video(str(video_path))
    scene_manager = SceneManager()
    scene_manager.add_detector(ContentDetector(threshold=SCENE_THRESHOLD, min_scene_len=MIN_SCENE_LENGTH))
    scene_manager.detect_scenes(video)
    scene_list = scene_manager.get_scene_list()

    scenes = []
    for idx, (start, end) in enumerate(scene_list):
        scenes.append(Scene(
            scene_id=f"{video_id}_s{idx:04d}",
            video_id=video_id,
            scene_index=idx,
            start_frame=start.get_frames(),
            end_frame=end.get_frames(),
            start_time=start.get_seconds(),
            end_time=end.get_seconds(),
            duration=end.get_seconds() - start.get_seconds()
        ))
    return scenes

def extract_keyframes(video_path: Path, scenes: List[Scene], video_id: str) -> List[Scene]:
    output_dir = SCENES_DIR / video_id
    output_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return scenes

    for scene in scenes:
        mid_frame = (scene.start_frame + scene.end_frame) // 2
        cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame)
        ret, frame = cap.read()
        if ret:
            keyframe_path = output_dir / f"{scene.scene_id}.jpg"
            cv2.imwrite(str(keyframe_path), frame)
            scene.keyframe_path = str(keyframe_path.relative_to(BASE_DIR))

    cap.release()
    return scenes

def encode_image_base64(image_path: Path) -> str:
    with open(image_path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode("utf-8")

def call_claude_vision(image_path: Path, prompt: str) -> Optional[str]:
    try:
        image_data = encode_image_base64(image_path)
        message = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=256,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": image_data}},
                    {"type": "text", "text": prompt}
                ]
            }]
        )
        return message.content[0].text.strip()
    except Exception as e:
        print(f"Claude API error: {e}")
        return None

def parse_comma_list(text: str) -> List[str]:
    if not text:
        return []
    return [item.strip().lower() for item in text.split(',') if item.strip()]

def label_keyframe(image_path: Path, video_id: str, scene_id: str) -> Optional[Annotation]:
    image_id = hashlib.md5(str(image_path).encode()).hexdigest()[:12]

    mood_response = call_claude_vision(image_path, MOOD_PROMPT)
    time.sleep(API_RATE_LIMIT_DELAY)
    style_response = call_claude_vision(image_path, STYLE_PROMPT)
    time.sleep(API_RATE_LIMIT_DELAY)
    object_response = call_claude_vision(image_path, OBJECT_PROMPT)

    if not all([mood_response, style_response, object_response]):
        return None

    return Annotation(
        image_id=image_id,
        video_id=video_id,
        scene_id=scene_id,
        mood_words=parse_comma_list(mood_response),
        style_words=parse_comma_list(style_response),
        object_words=parse_comma_list(object_response),
        labeling_model=CLAUDE_MODEL,
        prompt_version=PROMPT_VERSION
    )

def save_annotation(annotation: Annotation):
    path = ANNOTATIONS_DIR / f"{annotation.image_id}.json"
    with open(path, 'w') as f:
        json.dump(asdict(annotation), f, indent=2)

def load_annotation(image_id: str) -> Optional[Annotation]:
    path = ANNOTATIONS_DIR / f"{image_id}.json"
    if path.exists():
        with open(path, 'r') as f:
            return Annotation(**json.load(f))
    return None

def generate_mood_embedding(mood_words: List[str]) -> List[float]:
    if not mood_words:
        return [0.0] * 384
    embeddings = sbert_model.encode(mood_words)
    return np.mean(embeddings, axis=0).tolist()

print("Part 1 functions defined!")

In [ ]:
# Load video manifest
manifest_path = BASE_DIR / "video_manifest.csv"
manifest_df = pd.read_csv(manifest_path)
manifest_df['local_path'] = manifest_df['local_path'].str.replace('\\\\', '/', regex=False).str.replace('\\', '/', regex=False)
manifest_df['file_path'] = manifest_df['local_path']

print(f"Loaded {len(manifest_df)} videos")
display(manifest_df)

In [ ]:
# Part 1 실행: 비디오 처리 파이프라인
def run_part1(manifest_df: pd.DataFrame, skip_existing: bool = True):
    all_results = []
    available_videos = manifest_df[manifest_df['download_ok'] == True]

    for idx, row in available_videos.iterrows():
        video_id = row['video_id']
        video_path = BASE_DIR / row['file_path']

        print(f"\n{'='*60}")
        print(f"Processing: {video_id}")
        print(f"{'='*60}")

        if not video_path.exists():
            print(f"Video not found: {video_path}")
            continue

        # Scene detection
        print("[1/4] Detecting scenes...")
        scenes = detect_scenes(video_path, video_id)
        print(f"  Found {len(scenes)} scenes")

        if not scenes:
            continue

        # Keyframe extraction
        print("[2/4] Extracting keyframes...")
        scenes = extract_keyframes(video_path, scenes, video_id)

        # Save scene metadata
        scene_meta_path = SCENES_DIR / video_id / "scene_metadata.json"
        with open(scene_meta_path, 'w') as f:
            json.dump([asdict(s) for s in scenes], f, indent=2)

        # Labeling
        print("[3/4] Labeling with Claude API...")
        for scene in tqdm(scenes, desc="Labeling"):
            if not scene.keyframe_path:
                continue

            keyframe_path = BASE_DIR / scene.keyframe_path
            image_id = hashlib.md5(str(keyframe_path).encode()).hexdigest()[:12]

            if skip_existing:
                cached = load_annotation(image_id)
                if cached:
                    all_results.append((scene, cached))
                    continue

            annotation = label_keyframe(keyframe_path, video_id, scene.scene_id)
            if annotation:
                save_annotation(annotation)
                all_results.append((scene, annotation))

            time.sleep(API_RATE_LIMIT_DELAY)

    # Build dataset
    print("\n[4/4] Building GACS dataset...")
    dataset = []
    for scene, annotation in tqdm(all_results, desc="Building"):
        mood_vector = generate_mood_embedding(annotation.mood_words)
        dataset.append({
            'image_id': annotation.image_id,
            'video_id': annotation.video_id,
            'scene_id': annotation.scene_id,
            'keyframe_path': scene.keyframe_path or "",
            'mood_vector': ','.join(map(str, mood_vector)),
            'mood_words': ', '.join(annotation.mood_words),
            'style_words': ', '.join(annotation.style_words),
            'object_words': ', '.join(annotation.object_words)
        })

    df = pd.DataFrame(dataset)
    output_path = EMBEDDINGS_DIR / "gacs_dataset.csv"
    df.to_csv(output_path, index=False)

    print(f"\nDataset saved to: {output_path}")
    print(f"Total entries: {len(df)}")
    return df

# 실행
# gacs_df = run_part1(manifest_df)

---
# Part 2: Video Generator
무드 벡터 클러스터링 및 GACS/Baseline 비디오 생성

---

In [ ]:
# Part 2 Data Classes
@dataclass
class SceneData:
    image_id: str
    video_id: str
    scene_id: str
    keyframe_path: str
    mood_vector: np.ndarray
    mood_words: List[str]
    style_words: List[str]

@dataclass
class GeneratedVideoMetadata:
    video_id: str
    group: str
    mood_vector: List[float]
    mood_words: List[str]
    prompt_used: str
    source_scenes: List[str]
    created_at: str
    duration: float

In [ ]:
# Part 2 Prompts
GACS_GENERATION_PROMPT = """You are a GACS video generator.
Given:
- target mood words: {mood_words}
- scene pool with mood embeddings

Task:
Select 5 to 8 scenes whose mood vectors are closest to target.
Arrange into a 30-second emotional narrative.

Rules:
- Maintain emotional continuity.
- Avoid abrupt visual jumps.
- No duplicate scenes.
Output:
Ordered list of scene IDs.

Available scenes (scene_id: mood_words):
{scene_list}

Respond with ONLY a comma-separated list of scene IDs in the order they should appear."""

BASELINE_PROMPT = """Generate a random but visually coherent sequence of scenes.
Ignore mood vectors.

Select 5 to 8 scenes from the pool below.
Prioritize visual variety and coherent transitions.

Available scenes (scene_id: style_words):
{scene_list}

Respond with ONLY a comma-separated list of scene IDs."""

In [ ]:
# Part 2 Functions
def load_gacs_dataset() -> List[SceneData]:
    dataset_path = EMBEDDINGS_DIR / "gacs_dataset.csv"
    if not dataset_path.exists():
        print("Dataset not found. Run Part 1 first.")
        return []

    df = pd.read_csv(dataset_path)
    scenes = []
    for _, row in df.iterrows():
        scenes.append(SceneData(
            image_id=row['image_id'],
            video_id=row['video_id'],
            scene_id=row['scene_id'],
            keyframe_path=row['keyframe_path'],
            mood_vector=np.array([float(x) for x in row['mood_vector'].split(',')]),
            mood_words=[w.strip() for w in row['mood_words'].split(',')],
            style_words=[w.strip() for w in row['style_words'].split(',')]
        ))
    print(f"Loaded {len(scenes)} scenes")
    return scenes

def cluster_mood_vectors(scenes: List[SceneData], k: int = DEFAULT_K_CLUSTERS):
    vectors = np.stack([s.mood_vector for s in scenes])
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(vectors)

    cluster_info = {}
    for i in range(k):
        cluster_scenes = [s for s, l in zip(scenes, labels) if l == i]
        all_mood_words = [w for s in cluster_scenes for w in s.mood_words]
        word_counts = pd.Series(all_mood_words).value_counts()

        cluster_info[i] = {
            'centroid': kmeans.cluster_centers_[i],
            'num_scenes': len(cluster_scenes),
            'scene_ids': [s.scene_id for s in cluster_scenes],
            'top_mood_words': word_counts.head(5).index.tolist()
        }

    return kmeans, cluster_info

def select_scenes_gacs(target_mood_words: List[str], scenes: List[SceneData], num_candidates: int = 30):
    target_set = set(w.lower() for w in target_mood_words)
    scored = [(s, len(target_set & set(w.lower() for w in s.mood_words))) for s in scenes]
    scored.sort(key=lambda x: x[1], reverse=True)
    candidates = [s[0] for s in scored[:num_candidates]]

    scene_list = "\n".join([f"{s.scene_id}: {', '.join(s.mood_words)}" for s in candidates])
    prompt = GACS_GENERATION_PROMPT.format(mood_words=', '.join(target_mood_words), scene_list=scene_list)

    try:
        response = client.messages.create(model=CLAUDE_MODEL, max_tokens=256, messages=[{"role": "user", "content": prompt}])
        scene_ids = [s.strip() for s in response.content[0].text.strip().split(',')]
        valid_ids = [s.scene_id for s in candidates]
        scene_ids = [sid for sid in scene_ids if sid in valid_ids][:MAX_SCENES]
        if len(scene_ids) < MIN_SCENES:
            for s in candidates:
                if s.scene_id not in scene_ids:
                    scene_ids.append(s.scene_id)
                if len(scene_ids) >= MIN_SCENES:
                    break
        return scene_ids, prompt
    except Exception as e:
        print(f"Error: {e}")
        return [s.scene_id for s in candidates[:MAX_SCENES]], prompt

def select_scenes_baseline(scenes: List[SceneData], num_candidates: int = 30):
    candidates = random.sample(scenes, min(num_candidates, len(scenes)))
    scene_list = "\n".join([f"{s.scene_id}: {', '.join(s.style_words)}" for s in candidates])
    prompt = BASELINE_PROMPT.format(scene_list=scene_list)

    try:
        response = client.messages.create(model=CLAUDE_MODEL, max_tokens=256, messages=[{"role": "user", "content": prompt}])
        scene_ids = [s.strip() for s in response.content[0].text.strip().split(',')]
        valid_ids = [s.scene_id for s in candidates]
        scene_ids = [sid for sid in scene_ids if sid in valid_ids][:MAX_SCENES]
        if len(scene_ids) < MIN_SCENES:
            for s in candidates:
                if s.scene_id not in scene_ids:
                    scene_ids.append(s.scene_id)
                if len(scene_ids) >= MIN_SCENES:
                    break
        return scene_ids, prompt
    except Exception as e:
        print(f"Error: {e}")
        return [s.scene_id for s in candidates[:MAX_SCENES]], prompt

def get_scene_time_range(scene_id: str) -> Tuple[float, float]:
    parts = scene_id.rsplit('_s', 1)
    if len(parts) != 2:
        return 0, 3
    video_id = parts[0]
    metadata_path = SCENES_DIR / video_id / "scene_metadata.json"
    if metadata_path.exists():
        with open(metadata_path, 'r') as f:
            for scene in json.load(f):
                if scene['scene_id'] == scene_id:
                    return scene['start_time'], scene['end_time']
    return 0, 3

def get_video_path(scene_id: str) -> Optional[Path]:
    parts = scene_id.rsplit('_s', 1)
    if len(parts) != 2:
        return None
    video_id = parts[0]
    video_row = manifest_df[manifest_df['video_id'] == video_id]
    if video_row.empty:
        return None
    video_path = BASE_DIR / video_row.iloc[0]['local_path']
    return video_path if video_path.exists() else None

def render_video(scene_ids: List[str], output_path: Path) -> Optional[float]:
    clips = []
    per_scene_duration = TARGET_DURATION / len(scene_ids)

    for scene_id in tqdm(scene_ids, desc="Loading clips"):
        video_path = get_video_path(scene_id)
        if not video_path:
            continue
        start_time, end_time = get_scene_time_range(scene_id)
        scene_duration = min(end_time - start_time, per_scene_duration + 1)
        try:
            clip = VideoFileClip(str(video_path)).subclip(start_time, start_time + scene_duration)
            clip = fadeout(clip, TRANSITION_DURATION)
            if clips:
                clip = fadein(clip, TRANSITION_DURATION)
            clips.append(clip)
        except Exception as e:
            print(f"Error loading {scene_id}: {e}")

    if not clips:
        return None

    final = concatenate_videoclips(clips, method="compose")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    final.write_videofile(str(output_path), fps=OUTPUT_FPS, codec='libx264', audio_codec='aac', logger='bar')

    duration = final.duration
    for clip in clips:
        clip.close()
    final.close()

    return duration

print("Part 2 functions defined!")

In [ ]:
# Part 2 실행
def run_part2():
    scenes_data = load_gacs_dataset()
    if not scenes_data:
        return [], []

    kmeans, cluster_info = cluster_mood_vectors(scenes_data)

    print("\nCluster Summary:")
    for i, info in cluster_info.items():
        print(f"  Cluster {i}: {info['num_scenes']} scenes - {info['top_mood_words']}")

    all_gacs = []
    all_baseline = []

    for cluster_id, info in cluster_info.items():
        # GACS video
        print(f"\n{'='*60}")
        print(f"Generating GACS video for Cluster {cluster_id}")
        scene_ids, prompt = select_scenes_gacs(info['top_mood_words'], scenes_data)
        video_id = f"gacs_cluster{cluster_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        output_path = GACS_OUTPUT_DIR / f"{video_id}.mp4"
        duration = render_video(scene_ids, output_path)

        if duration:
            meta = GeneratedVideoMetadata(
                video_id=video_id, group="gacs", mood_vector=info['centroid'].tolist(),
                mood_words=info['top_mood_words'], prompt_used=prompt,
                source_scenes=scene_ids, created_at=datetime.now().isoformat(), duration=duration
            )
            meta_dir = GACS_OUTPUT_DIR / video_id
            meta_dir.mkdir(exist_ok=True)
            shutil.copy(output_path, meta_dir / "video.mp4")
            with open(meta_dir / "metadata.json", 'w') as f:
                json.dump(asdict(meta), f, indent=2)
            all_gacs.append(meta)

        # Baseline video
        print(f"\nGenerating Baseline video {cluster_id}")
        scene_ids, prompt = select_scenes_baseline(scenes_data)
        video_id = f"baseline_{cluster_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        output_path = BASELINE_OUTPUT_DIR / f"{video_id}.mp4"
        duration = render_video(scene_ids, output_path)

        if duration:
            meta = GeneratedVideoMetadata(
                video_id=video_id, group="baseline", mood_vector=[],
                mood_words=[], prompt_used=prompt,
                source_scenes=scene_ids, created_at=datetime.now().isoformat(), duration=duration
            )
            meta_dir = BASELINE_OUTPUT_DIR / video_id
            meta_dir.mkdir(exist_ok=True)
            shutil.copy(output_path, meta_dir / "video.mp4")
            with open(meta_dir / "metadata.json", 'w') as f:
                json.dump(asdict(meta), f, indent=2)
            all_baseline.append(meta)

    print(f"\n{'='*60}")
    print(f"Generated {len(all_gacs)} GACS videos, {len(all_baseline)} baseline videos")
    return all_gacs, all_baseline

# 실행
# gacs_videos, baseline_videos = run_part2()

---
# Part 3: YouTube Experiment
유튜브 업로드 및 메트릭 수집

---

In [ ]:
# Part 3 Imports (Google API)
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaFileUpload

CLIENT_SECRETS_FILE = BASE_DIR / "client_secrets.json"
CREDENTIALS_FILE = BASE_DIR / "youtube_credentials.json"
SCOPES = [
    'https://www.googleapis.com/auth/youtube.upload',
    'https://www.googleapis.com/auth/youtube.readonly',
    'https://www.googleapis.com/auth/yt-analytics.readonly'
]

In [ ]:
# Part 3 Prompts
TITLE_PROMPT = """Generate a YouTube title.
Under 60 characters.
Emotional but not clickbait.
No emojis.

Context:
- Video mood: {mood_words}
- Video group: {group}

Respond with ONLY the title, nothing else."""

DESCRIPTION_PROMPT = """Generate a short YouTube description.
Max 2 sentences.
No hashtags.
No marketing language.

Context:
- Video mood: {mood_words}
- Video group: {group}

Respond with ONLY the description, nothing else."""

TAGS_PROMPT = """Generate 8 to 12 single-word tags based on mood and style.

Context:
- Video mood: {mood_words}
- Video style: cinematic, mixed

Respond with ONLY comma-separated tags, nothing else."""

In [ ]:
# Part 3 Functions
def get_authenticated_services():
    credentials = None
    if CREDENTIALS_FILE.exists():
        with open(CREDENTIALS_FILE, 'r') as f:
            credentials = Credentials.from_authorized_user_info(json.load(f), SCOPES)

    if not credentials or not credentials.valid:
        if credentials and credentials.expired and credentials.refresh_token:
            credentials.refresh(Request())
        else:
            if not CLIENT_SECRETS_FILE.exists():
                raise FileNotFoundError("client_secrets.json not found")
            flow = InstalledAppFlow.from_client_secrets_file(str(CLIENT_SECRETS_FILE), SCOPES)
            credentials = flow.run_local_server(port=8080)
        with open(CREDENTIALS_FILE, 'w') as f:
            f.write(credentials.to_json())

    youtube = build('youtube', 'v3', credentials=credentials)
    youtube_analytics = build('youtubeAnalytics', 'v2', credentials=credentials)
    return youtube, youtube_analytics

def generate_video_metadata(video_meta: Dict) -> Dict:
    group = video_meta.get('group', 'unknown')
    mood_str = ', '.join(video_meta.get('mood_words', [])) or 'varied'

    try:
        r = client.messages.create(model=CLAUDE_MODEL, max_tokens=64, messages=[{"role": "user", "content": TITLE_PROMPT.format(mood_words=mood_str, group=group)}])
        title = r.content[0].text.strip()[:60]
    except:
        title = f"GACS Video - {group.capitalize()}"

    time.sleep(0.5)

    try:
        r = client.messages.create(model=CLAUDE_MODEL, max_tokens=128, messages=[{"role": "user", "content": DESCRIPTION_PROMPT.format(mood_words=mood_str, group=group)}])
        description = r.content[0].text.strip()
    except:
        description = "A video created for research purposes."

    time.sleep(0.5)

    try:
        r = client.messages.create(model=CLAUDE_MODEL, max_tokens=128, messages=[{"role": "user", "content": TAGS_PROMPT.format(mood_words=mood_str)}])
        tags = [t.strip() for t in r.content[0].text.strip().split(',')][:12]
    except:
        tags = ['video', 'research', 'mood']

    return {'title': title, 'description': description, 'tags': tags}

def upload_video(youtube_service, video_path: Path, title: str, description: str, tags: List[str]) -> Optional[str]:
    if not video_path.exists():
        return None

    body = {
        'snippet': {'title': title, 'description': description, 'tags': tags, 'categoryId': '22'},
        'status': {'privacyStatus': 'unlisted', 'selfDeclaredMadeForKids': False}
    }
    media = MediaFileUpload(str(video_path), mimetype='video/mp4', resumable=True)

    try:
        request = youtube_service.videos().insert(part='snippet,status', body=body, media_body=media)
        response = None
        while response is None:
            status, response = request.next_chunk()
            if status:
                print(f"Upload: {int(status.progress() * 100)}%")
        print(f"Uploaded: https://www.youtube.com/watch?v={response['id']}")
        return response['id']
    except HttpError as e:
        print(f"Error: {e}")
        return None

def collect_video_metrics(youtube_service, youtube_analytics_service, video_id: str):
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')

    metrics = {'video_id': video_id, 'collected_at': datetime.now().isoformat()}

    try:
        r = youtube_service.videos().list(part='statistics', id=video_id).execute()
        if r['items']:
            stats = r['items'][0]['statistics']
            metrics['views'] = int(stats.get('viewCount', 0))
            metrics['likes'] = int(stats.get('likeCount', 0))
            metrics['comments'] = int(stats.get('commentCount', 0))
    except:
        pass

    try:
        r = youtube_analytics_service.reports().query(
            ids='channel==MINE', startDate=start_date, endDate=end_date,
            metrics='views,estimatedMinutesWatched,averageViewDuration,averageViewPercentage',
            filters=f'video=={video_id}'
        ).execute()
        if r.get('rows'):
            row = r['rows'][0]
            metrics['watchTime'] = row[1] if len(row) > 1 else 0
            metrics['averageViewDuration'] = row[2] if len(row) > 2 else 0
            metrics['averageViewPercentage'] = row[3] if len(row) > 3 else 0
    except:
        pass

    return metrics

def analyze_results(metrics_df: pd.DataFrame):
    latest = metrics_df.sort_values('collected_at').groupby('video_id').last().reset_index()
    gacs = latest[latest['group'] == 'gacs']
    baseline = latest[latest['group'] == 'baseline']

    print(f"\n{'='*60}")
    print("GACS vs Baseline Analysis")
    print(f"{'='*60}")
    print(f"GACS videos: {len(gacs)}, Baseline: {len(baseline)}")

    for metric in ['views', 'likes', 'watchTime', 'averageViewPercentage']:
        if metric in gacs.columns and metric in baseline.columns:
            g_mean = gacs[metric].mean()
            b_mean = baseline[metric].mean()
            lift = ((g_mean - b_mean) / b_mean * 100) if b_mean > 0 else 0
            print(f"{metric}: GACS={g_mean:.2f}, Baseline={b_mean:.2f}, Lift={lift:+.1f}%")

print("Part 3 functions defined!")

In [ ]:
# Part 3 실행
def run_part3_upload(youtube_service):
    youtube_map = []

    for output_dir, group in [(GACS_OUTPUT_DIR, 'gacs'), (BASELINE_OUTPUT_DIR, 'baseline')]:
        for video_dir in output_dir.iterdir():
            if video_dir.is_dir():
                meta_path = video_dir / "metadata.json"
                video_path = video_dir / "video.mp4"
                if meta_path.exists() and video_path.exists():
                    with open(meta_path, 'r') as f:
                        meta = json.load(f)

                    yt_meta = generate_video_metadata(meta)
                    yt_id = upload_video(youtube_service, video_path, yt_meta['title'], yt_meta['description'], yt_meta['tags'])

                    if yt_id:
                        youtube_map.append({'youtube_id': yt_id, 'local_video_id': meta['video_id'], 'group': group})
                    time.sleep(5)

    with open(EXPERIMENTS_DIR / "youtube_map.json", 'w') as f:
        json.dump(youtube_map, f, indent=2)
    print(f"Uploaded {len(youtube_map)} videos")
    return youtube_map

def run_part3_collect(youtube_service, youtube_analytics_service):
    map_path = EXPERIMENTS_DIR / "youtube_map.json"
    if not map_path.exists():
        print("No youtube_map.json found")
        return None

    with open(map_path, 'r') as f:
        youtube_map = json.load(f)

    all_metrics = []
    for entry in tqdm(youtube_map):
        m = collect_video_metrics(youtube_service, youtube_analytics_service, entry['youtube_id'])
        m['local_video_id'] = entry['local_video_id']
        m['group'] = entry['group']
        all_metrics.append(m)
        time.sleep(0.5)

    df = pd.DataFrame(all_metrics)
    metrics_path = EXPERIMENTS_DIR / "metrics.csv"
    if metrics_path.exists():
        df = pd.concat([pd.read_csv(metrics_path), df], ignore_index=True)
    df.to_csv(metrics_path, index=False)

    analyze_results(df)
    return df

# 실행
# youtube, youtube_analytics = get_authenticated_services()
# youtube_map = run_part3_upload(youtube)
# metrics_df = run_part3_collect(youtube, youtube_analytics)

---
# 전체 파이프라인 실행

아래 셀을 순서대로 실행하세요.

---

In [ ]:
# Step 1: Dataset Builder 실행
# gacs_df = run_part1(manifest_df)

In [ ]:
# Step 2: Video Generator 실행
# gacs_videos, baseline_videos = run_part2()

In [ ]:
# Step 3: YouTube 업로드 (client_secrets.json 필요)
# youtube, youtube_analytics = get_authenticated_services()
# youtube_map = run_part3_upload(youtube)

In [ ]:
# Step 4: 메트릭 수집 (업로드 후 24시간 이후 실행)
# metrics_df = run_part3_collect(youtube, youtube_analytics)